# Hybrid Quantum-Classical Neural Network

How it works,

1. **Classical Pre-processing**: $h = f_\text{cl}(x; w_c)$. Takes input data $x$ and turns it into a cleaner or more useful form $h$ before sending it to the Quantum part.

2. **Quantum Embedding + Variational Circuit**: $|\psi(h,\theta)\rangle = U_\theta U_\phi(h),|0\rangle$. The Quantum circuit takes processed data $h$ and encodes it into Quantum states, and the circuit has tunable settings $θ$ (weight like knobs), which are the trainable “weights” of the Quantum part.

3. **Quantum Measurement as Features**: $z_i = \langle \psi(h,\theta) | M_i | \psi(h,\theta)\rangle$. After the Quantum circuit runs, the  measurement results become numbers $z_i$. These numbers are the quantum layer’s output features, similar to how a layer in a neural network outputs a vector.

4. **Classical Post-processing**: $\hat{y} = g_\text{cl}(z; w_o)$. Classical layer that uses the features $z$ to make the final prediction.


### Why use Hybrid Networks?

It is to leverage quantum power while using efficient classical computations, to potentially solve problems faster than classical deep learning

1. **Feature‑map Evaluation**: A quantum circuit can represent a high‑dimensional feature map implicitly. If a classical model would need to compute or store those features explicitly, the quantum circuit can be more efficient.

2. **Sampling Complexity**: Some quantum circuits can generate distributions that are hard to sample classically.

3. **Kernel Evaluation**: Quantum kernel methods might estimate certain kernels more efficiently than classical methods (if those kernels are classically intractable).

In [14]:
! pip3 install qiskit qiskit-machine-learning qiskit_algorithms torch torchvision numpy --break-system-packages

In [15]:
import torch
import torch.nn as nn 
import torch.optim as optim 
from sklearn.datasets import make_moons 
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler 
from qiskit.primitives import StatevectorEstimator 
from qiskit.circuit. library import RealAmplitudes
from qiskit_machine_learning.neural_networks import EstimatorQNN 
from qiskit_machine_learning.connectors import TorchConnector

#### Step 1: Generate and Preprocess Data

Create a dataset, scale/normalize, and split into train/test. Then, convert to PyTorch format.

In [ ]:
X, y = make_moons(n_samples=100, noise=0.1, random_state=42)

# Normalize and split data
scaler = StandardScaler()
X_scaled = scaler. fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Convert to PyTorch tensors
X_train_torch = torch.tensor(X_train, dtype=torch.float32)
X_test_torch = torch.tensor(X_test, dtype=torch.float32)
y_train_torch = torch.tensor(y_train, dtype=torch.long)
y_test_torch = torch.tensor(y_test, dtype=torch.long)

#### Step 2: Define the Quantum Circuit (RealAmplitudes)

Build a parameterized quantum circuit (Ansatz $U_\theta$), which is the learnable Quantum layer

In [17]:
num_qubits = 2
quantum_circuit = RealAmplitudes(num_qubits, reps=3)

/var/folders/td/rtvtvvjn3x1gfqswnc7bb6hm0000gn/T/ipykernel_61451/1101268564.py:2: DeprecationWarning: The class ``qiskit.circuit.library.n_local.real_amplitudes.RealAmplitudes`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.real_amplitudes instead.
  quantum_circuit = RealAmplitudes(num_qubits, reps=3)


#### Step 3: Wrap the Quantum Circuit in an EstimatorQNN

Turn the circuit into a QNN object that outputs expectation values.

In [18]:
estimator = StatevectorEstimator()
qnn = EstimatorQNN(
    circuit=quantum_circuit,
    estimator=estimator,
    input_params=quantum_circuit.parameters[:2],
    weight_params=quantum_circuit.parameters[2:]
)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


#### Step 4: Convert Quantum Circuit to a PyTorch-Compatible Layer

Use `TorchConnector` (or equivalent) to make it a PyTorch module, to train the quantum layer with classical ML tooling.

In [19]:
quantum_layer = TorchConnector(qnn)

#### Step 5: Define the Hybrid Quantum-Classical Neural Network

Combine the Classical layers + Quantum layer + Classical output layer to create the Hybrid Quantum-Classical Neural Network.

In [20]:
class HybridQuantumNN(nn.Module):

    def __init__(self):
        super(HybridQuantumNN, self).__init__()
        self.fc1 = nn.Linear(2, 2)
        self.quantum = quantum_layer
        self.fc2 = nn.Linear(1, 2)

    def forward(self, x):
        x = torch.tanh(self.fc1(x))
        x = self.quantum(x)
        x = self.fc2(x)
        return x

#### Step 6: Initialize Model, Loss Function, and Optimizer

Set up the model, a loss (e.g., cross‑entropy/MSE), and an optimizer (e.g., Adam gradient descent).

In [21]:
model = HybridQuantumNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

#### Step 7: Train the Model

Run forward/backprop, update parameters over epochs, to learn weights (including quantum circuit parameters).

In [22]:
epochs = 50

for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train_torch)
    loss = criterion(outputs, y_train_torch)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

Epoch 10/50, Loss: 0.7434
Epoch 20/50, Loss: 0.6827
Epoch 30/50, Loss: 0.6477
Epoch 40/50, Loss: 0.6258
Epoch 50/50, Loss: 0.5987


#### Step 8: Evaluate the Model


In [23]:
with torch.no_grad():
    y_pred = model(X_test_torch).argmax(dim=1)
    accuracy = (y_pred == y_test_torch).float().mean()
    print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Accuracy: 95.00%


### Boltzmann Machine (BM)

Boltzmann Machine is a neural network that doesn’t predict labels directly but learns the probability distribution of data.

```
Visible units (data you see)
   ●   ●   ●
    \  |  /
     \ | /
     Hidden units
      ● ●
```

- Visible = input data (pixels, features, etc.)
- Hidden = learn latent patterns

It's based on the Energy-based learning concept, where instead of weights → activations → outputs, it defines an energy landscape:

```
Low energy → Likely pattern
High energy → Unlikely pattern
```

> Recap: Energy is just a scoring function: $E(x)$
> - Lower score = more probable.
> - Probability relation: $P(x) \propto e^{-E(x)}$
> 
> Process as follows:
> 1. Show real data → lower its energy
> 2. Generate fake states → raise their energy
> 3. Adjust weights
> 4. Repeat

For example,

```
Energy surface:

      /\        unlikely
     /  \
 ___/    \___
     ↓
  likely data
```

In Classical Computing, to learn probabilities, BM must sample many states, for example,

```
Try state 001
Try state 110
Try state 011
…
```

But with Quantum Computing, by utilizing the Superposition, it allows the BM to try many states at once,

```
State exploration:

000 → 001 → 011 → 111 → …
(one at a time)
```